# 06 — Embeddings and Hybrid Retrieval

**First Finance - Arnaud Demes**  
**Day 1 · 15:15–16:00 · 15 minutes deck + 30 minutes guided notebook**

Lesson 05 created provenance-preserving chunks. This laboratory keeps those exact
units and changes how they are represented, filtered, fused and reranked. The offline
path is deterministic; live Ollama and OpenAI modes use the shared embedding gateway.

## Learning objectives

By the end of the laboratory, you can:

1. interpret cosine similarity as a geometric ranking signal, not confidence;
2. apply company and period eligibility before retrieval;
3. combine lexical and dense ranks with reciprocal-rank fusion (RRF);
4. inspect a transparent numeric-aware reranker;
5. reproduce exact-number and cross-company failure modes; and
6. version the corpus, embedding index and maintained evidence set together.

**The deterministic offline laboratory success condition:** reranked hybrid retrieval
recovers all four maintained evidence tokens while the controlled dense and unfiltered
failures stay visible. **Live contract:** Live OpenAI and Ollama runs report observed recall
and verify only provider-invariant structural behavior: finite vectors and scores, valid
dimensions, metadata filtering, visible stages and complete provenance.

## Where this fits

```text
Lesson 05                  Lesson 06                              Lesson 07
parse + chunk  →  embed → pre-filter → retrieve → fuse → rerank  →  evaluate + trace
```

### Laboratory map

| Minutes | Work | Observable output |
|---:|---|---|
| 0–5 | Rebuild and version seven Lesson 05 chunks | pipeline + projection |
| 5–10 | Compare dense similarities | query-by-passage heatmap |
| 10–15 | Inspect lexical, dense and fused ranks | ranked ladders |
| 15–20 | Reproduce exact-term and leakage failures | two safety figures |
| 20–25 | Explain RRF and reranking | contribution figures |
| 25–28 | Score all four maintained questions | four-stage scorecard |
| 28–30 | Change one RRF weight and verify | moved-ranking marker |

In [ ]:
from __future__ import annotations

import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch

from finai_academy.chunking import contextualize_chunks, structure_aware_chunks
from finai_academy.documents import load_source_manifest, parse_html, parse_pdf
from finai_academy.hybrid_retrieval import (
    DenseIndex,
    DeterministicTeachingEmbeddings,
    IndexedPassage,
    KeywordIndex,
    RetrievalFilters,
)
from finai_academy.lesson_support import (
    compact_manifest_labels,
    normalize_rows,
    spread_label_positions,
)
from finai_academy.providers import check_provider_configuration, create_embeddings
from finai_academy.reranking import RERANK_FEATURE_WEIGHTS, rerank_candidates
from finai_academy.retrieval_pipeline import retrieve_evidence, verify_retrieval_runs
from finai_academy.settings import Settings

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "assets" / "course-data"
INDEX_ROOT = Path(os.getenv("FINAI_INDEX_DIR", tempfile.gettempdir())) / "finai-lesson06-index"

COLORS = {
    "navy": "#051C2A", "blue": "#1F40CB", "cyan": "#00A2EB",
    "orange": "#F07D00", "green": "#2E8B57", "grey": "#64748B",
    "light": "#E8EEF5", "red": "#C43D3D",
}
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.titleweight": "bold"})

def annotate_points_without_collisions(axis, annotations, *, minimum_gap=28):
    axis.figure.canvas.draw()
    axis_box = axis.get_window_extent()
    point_pixels = axis.transData.transform([item[0] for item in annotations])
    median_x = float(np.median(point_pixels[:, 0]))
    for side in (1, -1):
        selected = [
            (index, item, point_pixels[index])
            for index, item in enumerate(annotations)
            if (point_pixels[index, 0] <= median_x) == (side == 1)
        ]
        selected.sort(key=lambda entry: (entry[2][1], entry[1][1]))
        label_y = spread_label_positions(
            [entry[2][1] for entry in selected],
            lower=axis_box.y0 + 14, upper=axis_box.y1 - 14, minimum_gap=minimum_gap,
        )
        for (_index, (point, label, color), pixel), placed_y in zip(selected, label_y, strict=True):
            y_offset = (placed_y - pixel[1]) * 72 / axis.figure.dpi
            axis.annotate(
                label, point, xytext=(7 * side, y_offset), textcoords="offset points",
                fontsize=7.5, color=color, ha="left" if side == 1 else "right", va="center",
                arrowprops={"arrowstyle": "-", "color": COLORS["grey"], "lw": 0.6},
            )

def label_horizontal_scores(axis, bars, values, *, digits=3):
    lower = min(0.0, min(values, default=0.0))
    upper = max(0.0, max(values, default=0.0))
    span = max(upper - lower, 0.2)
    axis.set_xlim(lower - 0.06 * span, upper + 0.22 * span)
    for bar, value in zip(bars, values, strict=True):
        direction = 1 if value >= 0 else -1
        axis.text(
            value + direction * 0.018 * span, bar.get_y() + bar.get_height() / 2,
            f"{value:.{digits}f}", va="center",
            ha="left" if direction == 1 else "right", fontsize=8.2,
        )

live_mode = os.getenv("FINAI_LIVE_MODE", "0") == "1"
if live_mode:
    settings = Settings.from_environment()
    problems = check_provider_configuration(settings)
    if problems:
        raise RuntimeError(" ".join(problems))
    embeddings = create_embeddings(settings)
    embedding_provider = settings.embedding_provider
    embedding_model = settings.embedding_model
else:
    embeddings = DeterministicTeachingEmbeddings()
    embedding_provider = "offline"
    embedding_model = embeddings.model_name
print(f"Embedding runtime: {embedding_provider} / {embedding_model}")

### 1. One observable retrieval pipeline

Every stage has a different contract. Metadata filters decide eligibility before any
similarity score is computed. Lexical and dense channels then rank the same eligible
passages. RRF combines rank positions, and reranking adds transparent evidence features.

In [ ]:
stages = [
    ("Manifest", "verified sources"),
    ("Contextual chunks", "7 evidence units"),
    ("Pre-filter", "company + period"),
    ("Two channels", "TF-IDF + cosine"),
    ("RRF", "rank contributions"),
    ("Reranker", "evidence features"),
]
fig, ax = plt.subplots(figsize=(15, 3.8))
ax.set_xlim(0, 15)
ax.set_ylim(0, 3)
ax.axis("off")
for index, (title, subtitle) in enumerate(stages):
    x = 0.2 + index * 2.5
    if index:
        ax.annotate("", xy=(x - 0.12, 1.35), xytext=(x - 0.45, 1.35),
                    arrowprops={"arrowstyle": "->", "lw": 2.2, "color": COLORS["cyan"]})
    box = FancyBboxPatch((x, 0.75), 2.0, 1.2, boxstyle="round,pad=0.04",
                         facecolor=COLORS["navy"] if index in {2, 5} else "white",
                         edgecolor=COLORS["blue"], linewidth=2)
    ax.add_patch(box)
    color = "white" if index in {2, 5} else COLORS["navy"]
    ax.text(x + 1.0, 1.50, title, ha="center", va="center", weight="bold", color=color)
    ax.text(x + 1.0, 1.12, subtitle, ha="center", va="center", fontsize=8.5, color=color)
ax.set_title("Figure 1 — Eligibility precedes ranking; reranking follows fusion", pad=10)
plt.tight_layout()
plt.show()

In [ ]:
sources = load_source_manifest(DATA_ROOT / "manifest.json")
assert all(source.verify_fixture(REPO_ROOT) for source in sources)

contextual_chunks = []
for source in sources:
    fixture_path = REPO_ROOT / source.fixture_path
    blocks = parse_html(fixture_path, source) if fixture_path.suffix == ".html" else parse_pdf(fixture_path, source)
    structured = structure_aware_chunks(blocks, max_chars=220)
    contextual_chunks.extend(contextualize_chunks(structured))

assert len(contextual_chunks) == 7, "The versioned Lesson 05 policy must produce seven chunks."
passages = tuple(
    IndexedPassage(
        passage_id=chunk.chunk_id,
        company=chunk.company,
        period=chunk.period,
        document_type=chunk.document_type,
        section=" > ".join(chunk.section_path) or "Document",
        text=chunk.text,
        source_url=chunk.source_url,
    )
    for chunk in contextual_chunks
)
PASSAGE_LABELS = compact_manifest_labels(passages)

keyword_index = KeywordIndex(passages)
dense_index = DenseIndex(
    passages, embeddings, provider=embedding_provider, model=embedding_model,
    chunking_strategy="contextual-structure-v1-max220",
)
dense_index.save(INDEX_ROOT)

EVIDENCE_SET_VERSION = "lesson05-maintained-v1"
EXPECTED_EVIDENCE = {
    "nvda-data-center": {"question": "Which NVIDIA business generated $193.7 billion?", "company": "NVIDIA", "period": "FY2026", "token": "$193.7 billion"},
    "nvda-gaming-growth": {"question": "How fast did NVIDIA Gaming revenue grow?", "company": "NVIDIA", "period": "FY2026", "token": "41%"},
    "se-revenue": {"question": "What was Schneider Electric FY2025 revenue?", "company": "Schneider Electric", "period": "FY2025", "token": "EUR 40.2bn"},
    "se-margin": {"question": "What margin did Schneider adjusted EBITA reach?", "company": "Schneider Electric", "period": "FY2025", "token": "18.7%"},
}
print(f"Rebuilt {len(passages)} passages from manifest schema v1")
print(f"Maintained evidence set: {EVIDENCE_SET_VERSION}")
print(f"Index corpus hash: {dense_index.version.corpus_hash[:12]}…")

In [ ]:
document_matrix = dense_index.document_matrix
query_matrix = normalize_rows([embeddings.embed_query(item["question"]) for item in EXPECTED_EVIDENCE.values()])
combined = np.vstack([document_matrix, query_matrix])
centered = combined - combined.mean(axis=0, keepdims=True)
_u, _s, vt = np.linalg.svd(centered, full_matrices=False)
projection = centered @ vt[:2].T
passage_xy = projection[:len(passages)]
query_xy = projection[len(passages):]

fig, ax = plt.subplots(figsize=(12.5, 7.2))
company_colors = {"NVIDIA": COLORS["blue"], "Schneider Electric": COLORS["orange"]}
annotations = []
for index, point in enumerate(passage_xy):
    passage = passages[index]
    color = company_colors[passage.company]
    ax.scatter(*point, s=120, color=color, edgecolor="white", linewidth=1.3, zorder=3)
    annotations.append((point, PASSAGE_LABELS[passage.passage_id], color))
for index, question_id in enumerate(EXPECTED_EVIDENCE):
    ax.scatter(*query_xy[index], s=170, marker="*", color=COLORS["green"], edgecolor=COLORS["navy"], linewidth=0.8)
    annotations.append((query_xy[index], f"query · {question_id}", COLORS["green"]))
ax.margins(x=0.20, y=0.18)
annotate_points_without_collisions(ax, annotations)
ax.scatter([], [], s=90, color=COLORS["blue"], label="NVIDIA passage")
ax.scatter([], [], s=90, color=COLORS["orange"], label="Schneider passage")
ax.scatter([], [], s=130, marker="*", color=COLORS["green"], edgecolor=COLORS["navy"], label="maintained question")
ax.axhline(0, color=COLORS["light"], lw=1)
ax.axvline(0, color=COLORS["light"], lw=1)
ax.set_xlabel("SVD projection dimension 1 (teaching view only)")
ax.set_ylabel("SVD projection dimension 2 (teaching view only)")
ax.set_title(
    f"Figure 2 — 2D map, not retrieval score · 4 maintained queries · unfiltered · {embedding_provider}",
    loc="left",
)
ax.legend(frameon=False, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 2. Cosine similarity ranks directions

Cosine similarity is the dot product of normalized vectors. It measures directional
alignment in an embedding space. It is **not** a probability, factuality score or model
confidence. The projection above is only a visual approximation; the heatmap below uses
the full embedding dimension and raw cosine similarity.

In [ ]:
question_ids = list(EXPECTED_EVIDENCE)
passage_labels = [PASSAGE_LABELS[p.passage_id] for p in passages]
similarity_matrix = []
for item in EXPECTED_EVIDENCE.values():
    by_id = {passage.passage_id: score for passage, score in dense_index.cosine_scores(item["question"])}
    similarity_matrix.append([by_id[passage.passage_id] for passage in passages])
similarity_matrix = np.asarray(similarity_matrix)

observed_cosine_min = float(np.min(similarity_matrix))
observed_cosine_max = float(np.max(similarity_matrix))
cosine_padding = max(0.05, (observed_cosine_max - observed_cosine_min) * 0.08)
cosine_vmin = max(-1.0, observed_cosine_min - cosine_padding)
cosine_vmax = min(1.0, observed_cosine_max + cosine_padding)
cosine_midpoint = (cosine_vmin + cosine_vmax) / 2

fig, ax = plt.subplots(figsize=(16.0, 6.4))
image = ax.imshow(similarity_matrix, cmap="coolwarm", vmin=cosine_vmin, vmax=cosine_vmax, aspect="auto")
ax.set_xticks(range(len(passage_labels)), passage_labels, rotation=30, ha="right")
ax.set_yticks(range(len(question_ids)), question_ids)
for row in range(similarity_matrix.shape[0]):
    for column in range(similarity_matrix.shape[1]):
        value = similarity_matrix[row, column]
        text_color = "white" if abs(value - cosine_midpoint) > (cosine_vmax - cosine_vmin) * 0.30 else COLORS["navy"]
        ax.text(column, row, f"{value:.2f}", ha="center", va="center", color=text_color, weight="bold")
ax.set_title(
    f"Figure 3 — Full-dimensional raw cosine · 4 maintained queries · unfiltered · {embedding_provider}",
    loc="left",
)
fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03, label="raw cosine similarity (not confidence)")
plt.tight_layout()
plt.show()

In [ ]:
retrieval_runs = {}
for question_id, item in EXPECTED_EVIDENCE.items():
    retrieval_runs[question_id] = retrieve_evidence(
        item["question"], keyword_index=keyword_index, dense_index=dense_index,
        filters=RetrievalFilters(company=item["company"], period=item["period"]),
        candidate_k=4, final_k=2,
    )

example_id = "nvda-data-center"
example = EXPECTED_EVIDENCE[example_id]
run = retrieval_runs[example_id]
dense_cosine = {passage.passage_id: score for passage, score in dense_index.cosine_scores(example["question"], run.filters)}
ladder_data = [
    ("Keyword channel", "TF-IDF cosine score", [(hit.passage, hit.score) for hit in run.keyword_hits]),
    ("Dense channel", "raw embedding cosine similarity", [(hit.passage, dense_cosine[hit.passage.passage_id]) for hit in run.dense_hits]),
    ("Fusion", "RRF score (k=60)", [(hit.passage, hit.rrf_score) for hit in run.fused_hits]),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 6.2))
for axis, (title, xlabel, ranking) in zip(axes, ladder_data, strict=True):
    labels = [PASSAGE_LABELS[item.passage_id] for item, _score in ranking]
    scores = [score for _item, score in ranking]
    colors = [COLORS["green"] if example["token"] in item.text else COLORS["blue"] for item, _score in ranking]
    y = np.arange(len(ranking))
    bars = axis.barh(y, scores, color=colors)
    axis.set_yticks(y, labels)
    axis.invert_yaxis()
    axis.set_xlabel(xlabel)
    axis.set_title(title, loc="left")
    axis.spines[["top", "right"]].set_visible(False)
    label_horizontal_scores(axis, bars, scores)
fig.suptitle(
    f"Figure 4 — Query: {example['question']} · filter: {example['company']} · {example['period']}",
    weight="bold",
)
fig.text(0.5, 0.01, "Green = passage contains the maintained evidence token", ha="center", color=COLORS["green"])
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
plt.show()

## Failure lab

Two controlled failures establish why the pipeline needs more than a dense nearest
neighbor lookup:

1. the deterministic teaching embedding deliberately excludes numeric tokens, so an
   exact-number-only query collapses to a dense tie;
2. an unfiltered energy-management query selects Schneider Electric evidence even when
   the application scope is NVIDIA. Eligibility must be enforced before ranking.

In [ ]:
controlled_embeddings = DeterministicTeachingEmbeddings()
controlled_keyword_index = KeywordIndex(passages)
controlled_dense_index = DenseIndex(
    passages, controlled_embeddings, provider="controlled-offline",
    model=controlled_embeddings.model_name, chunking_strategy="contextual-structure-v1-max220",
)
exact_query = "18.7%"
keyword_exact = controlled_keyword_index.search(exact_query, top_k=4)
dense_exact = controlled_dense_index.cosine_scores(exact_query)[:4]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
exact_channels = [
    (axes[0], "Keyword", "TF-IDF cosine score", [(hit.passage, hit.score) for hit in keyword_exact]),
    (axes[1], "Dense", "raw embedding cosine similarity", dense_exact),
]
for axis, title, xlabel, items in exact_channels:
    labels = [PASSAGE_LABELS[p.passage_id] for p, _ in items]
    values = [score for _, score in items]
    colors = [COLORS["green"] if exact_query in p.text else COLORS["red"] for p, _ in items]
    y = np.arange(len(items))
    bars = axis.barh(y, values, color=colors)
    axis.set_yticks(y, labels)
    axis.invert_yaxis()
    axis.set_xlabel(xlabel)
    axis.set_title(title, loc="left")
    axis.spines[["top", "right"]].set_visible(False)
    label_horizontal_scores(axis, bars, values)
fig.suptitle(
    f"Figure 5 — Controlled query: {exact_query!r} · no metadata filter · lexical vs dense tie",
    weight="bold",
)
plt.tight_layout()
plt.show()

assert exact_query in keyword_exact[0].passage.text
assert exact_query not in dense_exact[0][0].text
assert all(score == 0.0 for _passage, score in dense_exact)
print("Dense exact-term failure reproduced")

### Pre-filtering is a safety boundary

Post-filtering a global top-k is unsafe: the correct company's evidence may never enter
the candidate set. `RetrievalFilters` constrains company, period, document type and
section before both lexical and dense ranking. If nothing is eligible, the orchestrator
abstains instead of silently broadening the search.

In [ ]:
leakage_query = "energy management organic growth"
controlled_unfiltered = controlled_dense_index.cosine_scores(leakage_query)[:4]
nvidia_filter = RetrievalFilters(company="NVIDIA", period="FY2026")
controlled_filtered = controlled_dense_index.cosine_scores(leakage_query, nvidia_filter)[:4]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for axis, title, items in [
    (axes[0], "Controlled · no metadata eligibility", controlled_unfiltered),
    (axes[1], "Controlled · pre-filter: NVIDIA · FY2026", controlled_filtered),
]:
    labels = [PASSAGE_LABELS[p.passage_id] for p, _ in items]
    values = [score for _, score in items]
    colors = [company_colors[p.company] for p, _ in items]
    y = np.arange(len(items))
    bars = axis.barh(y, values, color=colors)
    axis.set_yticks(y, labels)
    axis.invert_yaxis()
    axis.set_xlabel("raw embedding cosine similarity")
    axis.set_title(title, loc="left")
    axis.spines[["top", "right"]].set_visible(False)
    label_horizontal_scores(axis, bars, values)
fig.suptitle(
    f"Figure 6 — Controlled query: {leakage_query!r} · compare no filter vs NVIDIA · FY2026",
    weight="bold",
)
plt.tight_layout()
plt.show()

assert controlled_unfiltered[0][0].company == "Schneider Electric"
assert {passage.company for passage, _score in controlled_filtered} == {"NVIDIA"}
print("Cross-company leakage blocked")

### 3. Reciprocal-rank fusion combines positions, not incomparable scores

TF-IDF and embedding cosine scores do not share a calibrated scale. RRF avoids pretending
that they do. For channel weight `w`, constant `k=60` and one-based rank `r`, the
contribution is `w / (k + r)`. Shared passage IDs are deduplicated and their channel
contributions are added.

In [ ]:
fusion_hits = run.fused_hits[:4]
labels = [PASSAGE_LABELS[hit.passage.passage_id] for hit in fusion_hits]
keyword_parts = []
dense_parts = []
for hit in fusion_hits:
    ranks = dict(hit.channel_ranks)
    keyword_parts.append(1 / (60 + ranks["keyword"]) if "keyword" in ranks else 0)
    dense_parts.append(1 / (60 + ranks["dense"]) if "dense" in ranks else 0)

fig, ax = plt.subplots(figsize=(12.5, 6.0))
y = np.arange(len(labels))
ax.barh(y, keyword_parts, color=COLORS["blue"], label="keyword: 1 / (60 + rank)")
ax.barh(y, dense_parts, left=keyword_parts, color=COLORS["cyan"], label="dense: 1 / (60 + rank)")
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlabel("RRF score contribution")
ax.set_title(
    f"Figure 7 — RRF contributions · query: {example['question']} · filter: {example['company']} · {example['period']}",
    loc="left",
)
ax.legend(frameon=False, bbox_to_anchor=(1.0, 0.5), loc="center left")
ax.spines[["top", "right"]].set_visible(False)
label_horizontal_scores(ax, ax.patches[:len(fusion_hits)], [hit.rrf_score for hit in fusion_hits], digits=5)
plt.tight_layout()
plt.show()

### 4. Reranking makes the second-stage decision explicit

The deterministic reranker combines normalized lexical coverage (including ticker tokens
such as `NVDA`), exact numeric-literal coverage, section overlap, metadata eligibility and
the fusion signal. Numeric coverage carries the
largest weight in this financial laboratory. The resulting value is a **rerank score**,
not confidence.

In [ ]:
reranked_all = rerank_candidates(example["question"], run.fused_hits, top_k=len(run.fused_hits))
feature_names = list(RERANK_FEATURE_WEIGHTS)
feature_colors = [COLORS["blue"], COLORS["orange"], COLORS["cyan"], COLORS["green"], COLORS["grey"]]
labels = [PASSAGE_LABELS[hit.passage.passage_id] for hit in reranked_all]

fig, ax = plt.subplots(figsize=(13, 6.4))
left = np.zeros(len(reranked_all))
for name, color in zip(feature_names, feature_colors, strict=True):
    values = np.asarray([getattr(hit.features, name) * RERANK_FEATURE_WEIGHTS[name] for hit in reranked_all])
    ax.barh(np.arange(len(labels)), values, left=left, color=color, label=f"{name} × {RERANK_FEATURE_WEIGHTS[name]:.2f}")
    left += values
ax.set_yticks(np.arange(len(labels)), labels)
ax.invert_yaxis()
ax.set_xlabel("weighted rerank feature contribution (sum = rerank score)")
ax.set_title(
    f"Figure 8 — Rerank features · query: {example['question']} · filter: {example['company']} · {example['period']}",
    loc="left",
)
ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
label_horizontal_scores(ax, ax.patches[:len(reranked_all)], [hit.score for hit in reranked_all])
plt.tight_layout()
plt.show()

In [ ]:
def contains_expected(passage, expected):
    return expected["token"] in passage.text

stage_success = {"Keyword": [], "Dense": [], "RRF fusion": [], "Reranked hybrid": []}
for question_id, expected in EXPECTED_EVIDENCE.items():
    run = retrieval_runs[question_id]
    stage_success["Keyword"].append(contains_expected(run.keyword_hits[0].passage, expected))
    stage_success["Dense"].append(contains_expected(run.dense_hits[0].passage, expected))
    stage_success["RRF fusion"].append(contains_expected(run.fused_hits[0].passage, expected))
    stage_success["Reranked hybrid"].append(contains_expected(run.reranked_hits[0].passage, expected))
score_matrix = np.asarray(list(stage_success.values()), dtype=float)

fig, ax = plt.subplots(figsize=(12.5, 6.5))
image = ax.imshow(score_matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(question_ids)), question_ids, rotation=20, ha="right")
ax.set_yticks(range(len(stage_success)), list(stage_success))
for row in range(score_matrix.shape[0]):
    for column in range(score_matrix.shape[1]):
        passed = bool(score_matrix[row, column])
        ax.text(column, row, "PASS" if passed else "MISS", ha="center", va="center",
                color="white" if passed else COLORS["red"], weight="bold")
    ax.text(score_matrix.shape[1] - 0.05, row, f"  {int(score_matrix[row].sum())}/4", va="center", fontsize=9)
ax.set_title(f"Figure 9 — Four-stage maintained-evidence scorecard · {EVIDENCE_SET_VERSION}", loc="left")
fig.colorbar(image, ax=ax, fraction=0.025, pad=0.08, label="expected evidence recovered at rank 1")
plt.tight_layout()
plt.show()

maintained_recall = {stage: sum(values) / len(values) for stage, values in stage_success.items()}
if not live_mode:
    assert maintained_recall["Reranked hybrid"] > maintained_recall["Dense"]
    print("Hybrid retrieval improves maintained recall")
else:
    print("Live provider recall observed — no provider-specific ranking threshold asserted")
print(pd.Series(maintained_recall, name="recall@1").to_string())

## Verification

The laboratory passes only when source provenance, index versioning, eligibility, the
two controlled failures and maintained evidence recovery are all observable. A high
similarity score alone is never the acceptance criterion.

In [ ]:
provider_filtered = dense_index.cosine_scores(leakage_query, nvidia_filter)
verification_report = verify_retrieval_runs(
    retrieval_runs,
    expected_evidence={question_id: item["token"] for question_id, item in EXPECTED_EVIDENCE.items()},
    require_expected_evidence=not live_mode,
    required_artifacts=(INDEX_ROOT / "manifest.json", INDEX_ROOT / "vectors.npy"),
)
checks = dict(verification_report.checks)
checks.update({
    "seven manifest-derived passages": len(passages) == 7,
    "provider vectors are finite": dense_index.dimension > 0 and document_matrix.shape == (len(passages), dense_index.dimension) and np.isfinite(document_matrix).all(),
    "query rows normalized before projection": np.isfinite(query_matrix).all() and np.allclose(np.linalg.norm(query_matrix, axis=1), 1.0),
    "provider filter blocks cross-company passages": bool(provider_filtered) and {p.company for p, _score in provider_filtered} == {"NVIDIA"},
})
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {label}")
assert all(checks.values())
print("PASS — hybrid retrieval laboratory verified")

## Challenge

Increase only the keyword RRF weight from `1.0` to `3.0`, keep the dense weight at
`1.0`, and identify which maintained question rankings move. Explain why a moved rank
is a tuning observation rather than proof that the new weight is better.

In [ ]:
moved_rankings = []
for question_id, expected in EXPECTED_EVIDENCE.items():
    baseline_ids = [hit.passage.passage_id for hit in retrieval_runs[question_id].fused_hits]
    weighted = retrieve_evidence(
        expected["question"], keyword_index=keyword_index, dense_index=dense_index,
        filters=RetrievalFilters(company=expected["company"], period=expected["period"]),
        candidate_k=4, final_k=2, weights={"keyword": 3.0, "dense": 1.0},
    )
    weighted_ids = [hit.passage.passage_id for hit in weighted.fused_hits]
    if weighted_ids != baseline_ids:
        moved_rankings.append(question_id)
        baseline_order = " > ".join(passage_id.rsplit("-", 1)[-1] for passage_id in baseline_ids)
        weighted_order = " > ".join(passage_id.rsplit("-", 1)[-1] for passage_id in weighted_ids)
        print(f"Ranking moved — {question_id}: {baseline_order} → {weighted_order}")
if not live_mode:
    assert moved_rankings
if moved_rankings:
    print("Questions with a moved ranking:", ", ".join(moved_rankings))
else:
    print("No rankings moved with the selected live provider; the result is still valid.")

## Capstone integration

The Financial Analyst Copilot now gains a provider-neutral retrieval boundary:

```text
versioned contextual chunks
  → versioned embedding index
  → company/period pre-filter
  → lexical + dense candidate rankings
  → reciprocal-rank fusion
  → transparent evidence reranker
  → final provenance-preserving passages
```

The index manifest binds provider, model, dimension, chunking strategy, corpus hash and
ordered passage IDs. A mismatch must trigger a rebuild, never silent vector reuse.

## Recap

- Embeddings create geometric ranking signals; cosine similarity is not confidence.
- Metadata pre-filtering defines eligibility and prevents cross-company leakage.
- Lexical retrieval protects exact terms that dense representations may omit.
- RRF combines rank positions without pretending channel scores are calibrated.
- Reranking must expose its evidence features and score type.
- Corpus, chunking, embedding model and expected evidence are versioned together.

**Next — Lesson 07:** turn these four maintained questions into evaluation cases, attach
stage-level traces, and distinguish retrieval quality from answer groundedness.